# 第十章 智能体通信协议

在前面的章节中，我们构建了功能完备的单体智能体，它们具备推理、工具调用和记忆能力。然而，当我们尝试构建更复杂的 AI 系统时，自然会有疑问：**如何让智能体与外部世界高效交互？如何让多个智能体相互协作？**

这正是智能体通信协议要解决的核心问题。本章将为 HelloAgents 框架引入三种通信协议：**MCP（Model Context Protocol）** 用于智能体与工具的标准化通信，**A2A（Agent-to-Agent Protocol）** 用于智能体间的点对点协作，**ANP（Agent Network Protocol）** 用于构建大规模智能体网络。这三种协议共同构成了智能体通信的基础设施层。

通过本章的学习，您将掌握智能体通信协议的设计理念和实践技能，理解三种主流协议的设计差异，学会如何选择合适的协议来解决实际问题。

## 10.1 智能体通信协议基础

### 10.1.1 为何需要通信协议

回顾我们在第七章构建的 ReAct 智能体，它已经具备了强大的推理和工具调用能力。让我们看一个典型的使用场景，以及它面临的根本性限制：

1. **工具集成的困境**：每当需要访问新的外部服务（如 GitHub API、数据库、文件系统），都必须编写专门的 Tool 类，工作量大且不同开发者编写的工具无法互相兼容。
2. **能力扩展的瓶颈**：智能体的能力被限制在预先定义的工具集内，无法动态发现和使用新的服务。
3. **协作的缺失**：当任务复杂到需要多个专业智能体协作时，只能通过手动编排来协调。

**通信协议的核心价值**：提供标准化接口规范，让智能体以统一方式访问各种外部服务，就像互联网的 TCP/IP 协议统一了设备间的通信方式。

In [2]:
from hello_agents import ReActAgent, HelloAgentsLLM
from hello_agents.tools import CalculatorTool, SearchTool

llm = HelloAgentsLLM()
agent = ReActAgent(name="AI助手", llm=llm)
agent.add_tool(CalculatorTool())
agent.add_tool(SearchTool())

# 智能体可以独立完成任务
response = agent.run("搜索最新的AI新闻，并计算相关公司的市值总和")

TypeError: ReActAgent.__init__() missing 1 required positional argument: 'tool_registry'

In [ ]:
# 传统方式：手动集成每个服务（存在的问题）
from hello_agents.tools import BaseTool

class GitHubTool(BaseTool):
    """需要手写GitHub API适配器"""
    def run(self, repo_url):
        # 大量的API调用代码...
        pass

class DatabaseTool(BaseTool):
    """需要手写数据库适配器"""
    def run(self, query):
        # 数据库连接和查询代码...
        pass

class WeatherTool(BaseTool):
    """需要手写天气API适配器"""
    def run(self, location):
        # 天气API调用代码...
        pass

# 每个新服务都需要重复这个过程
agent.add_tool(GitHubTool())
agent.add_tool(DatabaseTool())
agent.add_tool(WeatherTool())

In [ ]:
from hello_agents.tools import MCPTool

# 有了通信协议，代码大幅简化
mcp_tool = MCPTool()  # 内置服务器提供基础工具

# 或者连接到专业的MCP服务器
github_mcp = MCPTool(server_command=["npx", "-y", "@modelcontextprotocol/server-github"])
database_mcp = MCPTool(server_command=["python", "database_mcp_server.py"])

# 智能体自动获得所有能力，无需手写适配器
agent.add_tool(mcp_tool)
agent.add_tool(github_mcp)
agent.add_tool(database_mcp)

### 10.1.2 三种协议设计理念比较

**(1) MCP：智能体与工具的桥梁**

MCP（Model Context Protocol）由 Anthropic 团队提出，核心理念是**标准化智能体与外部工具/资源的通信方式**。设计哲学是"上下文共享"——不仅传递工具调用结果，还共享代码结构、依赖关系、提交历史等丰富上下文信息。

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-1.png" alt="" width="85%"/>
  <p>图 10.1 MCP 设计思想</p>
</div>

**(2) A2A：智能体间的对话**

A2A（Agent-to-Agent Protocol）由 Google 团队提出，核心理念是**实现智能体之间的点对点通信**。设计哲学是"对等通信"——每个智能体既是服务提供者，也是服务消费者，避免了中心化协调器的瓶颈。

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-2.png" alt="" width="85%"/>
  <p>图 10.2 A2A 设计思想</p>
</div>

**(3) ANP：智能体网络的基础设施**

ANP（Agent Network Protocol）是开源社区维护的概念性协议框架，核心理念是**构建大规模智能体网络的基础设施**。设计哲学是"去中心化服务发现"——通过服务注册、发现和路由机制，让智能体动态发现网络中的其他服务。

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-3.png" alt="" width="85%"/>
  <p>图 10.3 ANP 设计思想</p>
</div>

<div align="center">
  <p>表 10.1 三种协议对比</p>
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-table-1.png" alt="" width="85%"/>
</div>

**如何选择合适的协议？**
- 智能体需要访问外部服务（文件、数据库、API）→ 选择 **MCP**
- 需要多个智能体相互协作完成任务 → 选择 **A2A**
- 构建大规模的智能体生态系统 → 考虑 **ANP**

### 10.1.3 HelloAgents 通信协议架构设计

HelloAgents 的通信协议架构采用**三层设计**：

1. **协议实现层**：MCP 基于 FastMCP 库，A2A 基于 Google 官方 a2a-sdk，ANP 为自研轻量级实现
2. **工具封装层**：MCPTool、A2ATool、ANPTool 统一继承自 BaseTool，提供一致的 `run()` 方法
3. **智能体集成层**：所有智能体通过 Tool System 使用协议工具，无需关心底层协议细节

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-4.png" alt="" width="85%"/>
  <p>图 10.4 HelloAgents 通信协议设计</p>
</div>

```
hello_agents/
├── protocols/
│   ├── mcp/
│   │   ├── client.py       # MCP客户端（支持5种传输方式）
│   │   ├── server.py       # MCP服务器（FastMCP封装）
│   │   └── utils.py        # 工具函数
│   ├── a2a/
│   │   └── implementation.py  # A2A服务器/客户端
│   └── anp/
│       └── implementation.py  # ANP服务发现/注册
└── tools/builtin/
    └── protocol_tools.py   # 协议工具包装器（MCPTool/A2ATool/ANPTool）
```

### 10.1.4 本章学习目标与快速体验

在开始实战之前，先准备好开发环境：

In [ ]:
# 安装HelloAgents框架（第10章版本）
# !pip install "hello-agents[protocol]==0.2.2"
# 安装NodeJS，可以参考 Additional-Chapter 中的文档

from hello_agents.tools import MCPTool, A2ATool, ANPTool

# 1. MCP：访问工具
mcp_tool = MCPTool()
result = mcp_tool.run({
    "action": "call_tool",
    "tool_name": "add",
    "arguments": {"a": 10, "b": 20}
})
print(f"MCP计算结果: {result}")  # 输出: 30.0

# 2. ANP：服务发现
anp_tool = ANPTool()
anp_tool.run({
    "action": "register_service",
    "service_id": "calculator",
    "service_type": "math",
    "endpoint": "http://localhost:8080"
})
services = anp_tool.run({"action": "discover_services"})
print(f"发现的服务: {services}")

# 3. A2A：智能体通信
a2a_tool = A2ATool("http://localhost:5000")
print("A2A工具创建成功")

🧠 使用内存传输: HelloAgents-BuiltinServer
🔗 连接到 MCP 服务器...


INFO:mcp.server.lowlevel.server:Processing request of type ListToolsRequest
INFO:mcp.server.lowlevel.server:Processing request of type CallToolRequest
INFO:mcp.server.lowlevel.server:Processing request of type ListToolsRequest


✅ 连接成功！
🔌 连接已断开
🧠 使用内存传输: HelloAgents-BuiltinServer
🔗 连接到 MCP 服务器...
✅ 连接成功！
🔌 连接已断开
MCP计算结果: 工具 'add' 执行结果:
30.0
发现的服务: 找到 1 个服务:

服务ID: calculator
  名称: calculator
  类型: math
  端点: http://localhost:8080


A2A工具创建成功


## 10.2 MCP 协议实战

### 10.2.1 MCP 协议概念介绍

**(1) MCP：智能体的"USB-C"**

MCP 统一了智能体与外部工具的交互方式，就像 USB-C 统一了各种设备的连接方式。无论使用 Claude、GPT 还是其他模型，只要支持 MCP 协议，就能无缝访问相同的工具和资源。

**(2) MCP 三层架构（Host / Client / Server）**

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-5.png" alt="" width="85%"/>
  <p>图 10.5 MCP 案例演示</p>
</div>

- **Host（宿主层）**：如 Claude Desktop，负责接收用户提问并与模型交互
- **Client（客户端层）**：负责与 MCP Server 建立连接，发送请求并接收响应
- **Server（服务器层）**：执行实际操作（如文件扫描），返回结果

**(3) MCP 三大核心能力**

<div align="center">
  <p>表 10.2 MCP 核心能力</p>
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-table-2.png" alt="" width="85%"/>
</div>

- **Tools**：主动执行操作
- **Resources**：被动提供数据
- **Prompts**：指导性模板

**(4) MCP 工作流程**

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-6.png" alt="" width="85%"/>
  <p>图 10.6 MCP 工作流程</p>
</div>

**(5) MCP 与 Function Calling 的差异**

<div align="center">
  <p>表 10.3 Function Calling 与 MCP 对比</p>
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-table-3.png" alt="" width="85%"/>
</div>

> Function Calling 相当于你学会了"如何打电话"；MCP 是全球统一的"电话通信标准"。两者是互补关系，而非竞争关系。

In [3]:
import json

# ── 方式1：使用 Function Calling（需要为每个LLM单独定义）──

# OpenAI格式
openai_tools = [{"type": "function", "function": {"name": "search_github", "description": "搜索GitHub仓库", "parameters": {"type": "object", "properties": {"query": {"type": "string", "description": "搜索关键词"}}, "required": ["query"]}}}]

# Claude格式（注意：字段名是input_schema而非parameters）
claude_tools = [{"name": "search_github", "description": "搜索GitHub仓库", "input_schema": {"type": "object", "properties": {"query": {"type": "string", "description": "搜索关键词"}}, "required": ["query"]}}]

# ── 方式2：使用 MCP（统一接口，与模型无关）──
import asyncio
from hello_agents.protocols import MCPClient

async def demo_mcp_vs_function_calling():
    github_client = MCPClient(["npx", "-y", "@modelcontextprotocol/server-github"])
    async with github_client:
        tools = await github_client.list_tools()
        result = await github_client.call_tool("search_repositories", {"query": "AI agents"})
        print(f"MCP调用结果: {result}")

await demo_mcp_vs_function_calling()

📝 使用 Stdio 传输 (命令): npx -y @modelcontextprotocol/server-github
🔗 连接到 MCP 服务器...


RuntimeError: Client failed to connect: [Errno 2] No such file or directory: 'npx'

### 10.2.2 使用 MCP 客户端

HelloAgents 基于 FastMCP 2.0 实现了完整的 MCP 客户端功能，提供异步和同步两种 API。

In [3]:
import os
import asyncio
from hello_agents.protocols import MCPClient

# ── (1) 连接到 MCP 服务器 ──
async def connect_to_server():
    # 连接到社区提供的文件系统服务器，npx自动安装并运行服务器包(一个用Node.js写的程序)，启动一个子进程与mcp服务器通信
    client = MCPClient(["npx", "-y", "@modelcontextprotocol/server-filesystem", "."])
    #with块开始时，调用client的__aenter__方法建立连接，退出时调用client的__aexit__方法关闭连接
    async with client:  
        tools = await client.list_tools()
        print(f"可用工具: {[t['name'] for t in tools]}")

    # 连接到自定义的 Python MCP 服务器
    # client = MCPClient(["python", "my_mcp_server.py"])

# ── (2) 发现可用工具 ──
async def discover_tools():
    client = MCPClient(["npx", "-y", "@modelcontextprotocol/server-filesystem", "."])
    async with client:
        tools = await client.list_tools()
        print(f"服务器提供了 {len(tools)} 个工具：")
        for tool in tools:
            print(f"\n工具名称: {tool['name']}")
            print(f"描述: {tool.get('description', '无描述')}")
            if 'inputSchema' in tool:
                schema = tool['inputSchema']
                if 'properties' in schema:
                    print("参数:")
                    for param_name, param_info in schema['properties'].items():
                        print(f"  - {param_name} ({param_info.get('type','any')}): {param_info.get('description','')}")

await connect_to_server()
await discover_tools()

📝 使用 Stdio 传输 (命令): npx -y @modelcontextprotocol/server-filesystem .
🔗 连接到 MCP 服务器...
✅ 连接成功！
可用工具: ['read_file', 'read_text_file', 'read_media_file', 'read_multiple_files', 'write_file', 'edit_file', 'create_directory', 'list_directory', 'list_directory_with_sizes', 'directory_tree', 'move_file', 'search_files', 'get_file_info', 'list_allowed_directories']
🔌 连接已断开
📝 使用 Stdio 传输 (命令): npx -y @modelcontextprotocol/server-filesystem .
🔗 连接到 MCP 服务器...
✅ 连接成功！
服务器提供了 14 个工具：

工具名称: read_file
描述: Read the complete contents of a file as text. DEPRECATED: Use read_text_file instead.

工具名称: read_text_file
描述: Read the complete contents of a file from the file system as text. Handles various text encodings and provides detailed error messages if the file cannot be read. Use this tool when you need to examine the contents of a single file. Use the 'head' parameter to read only the first N lines of a file, or the 'tail' parameter to read only the last N lines of a file. Operates on the file as t

In [5]:
import asyncio
from hello_agents.protocols import MCPClient

# ── (3) 调用工具 ──
async def use_tools():
    client = MCPClient(["npx", "-y", "@modelcontextprotocol/server-filesystem", "."])
    async with client:
        result = await client.call_tool("read_file", {"path": "README.md"})
        print(f"文件内容：\n{result}")
        result = await client.call_tool("list_directory", {"path": "."})
        print(f"当前目录文件：{result}")
        result = await client.call_tool("write_file", {"path": "output.txt", "content": "Hello from MCP!"})
        print(f"写入结果：{result}")

# ── (4) 安全调用方式 ──
async def safe_tool_call():
    client = MCPClient(["npx", "-y", "@modelcontextprotocol/server-filesystem", "."])
    async with client:
        try:
            result = await client.call_tool("read_file", {"path": "nonexistent.txt"})
            print(result)
        except Exception as e:
            print(f"工具调用失败: {e}")

# ── (5) 访问资源 & 使用提示模板（需要服务器支持）──
async def access_resources_and_prompts(client):
    # 资源
    resources = client.list_resources()
    print(f"可用资源：{[r['uri'] for r in resources]}")
    resource_content = client.read_resource("file:///path/to/resource")
    print(f"资源内容：{resource_content}")
    # 提示模板
    prompts = client.list_prompts()
    print(f"可用提示：{[p['name'] for p in prompts]}")
    prompt = client.get_prompt("code_review", {"language": "python"})
    print(f"提示内容：{prompt}")

await use_tools()
await safe_tool_call()

📝 使用 Stdio 传输 (命令): npx -y @modelcontextprotocol/server-filesystem .
🔗 连接到 MCP 服务器...
✅ 连接成功！
🔌 连接已断开


ToolError: ENOENT: no such file or directory, open '/root/workspace/hello-agents/docs/chapter10/README.md'

In [6]:
"""
完整示例：使用 GitHub MCP 服务

注意：需要设置环境变量
    Linux/macOS: export GITHUB_PERSONAL_ACCESS_TOKEN="your_token_here"
    Windows: $env:GITHUB_PERSONAL_ACCESS_TOKEN="your_token_here"
"""
from dotenv import load_dotenv
from hello_agents.tools import MCPTool

load_dotenv()  # 加载环境变量
# 创建 GitHub MCP 工具
github_tool = MCPTool(
    server_command=["npx", "-y", "@modelcontextprotocol/server-github"]
)

# 1. 列出可用工具
print("📋 可用工具：")
result = github_tool.run({"action": "list_tools"})
print(result)

# 2. 搜索仓库
print("\n🔍 搜索仓库：")
result = github_tool.run({
    "action": "call_tool",
    "tool_name": "search_repositories",
    "arguments": {
        "query": "AI agents language:python",
        "page": 1,
        "perPage": 3
    }
})
print(result)

🔑 自动加载环境变量: GITHUB_PERSONAL_ACCESS_TOKEN
📝 使用 Stdio 传输 (命令): npx -y @modelcontextprotocol/server-github
🔗 连接到 MCP 服务器...
✅ 连接成功！
🔌 连接已断开
📋 可用工具：
📝 使用 Stdio 传输 (命令): npx -y @modelcontextprotocol/server-github
🔗 连接到 MCP 服务器...
✅ 连接成功！
🔌 连接已断开
找到 26 个工具:
- create_or_update_file: Create or update a single file in a GitHub repository
- search_repositories: Search for GitHub repositories
- create_repository: Create a new GitHub repository in your account
- get_file_contents: Get the contents of a file or directory from a GitHub repository
- push_files: Push multiple files to a GitHub repository in a single commit
- create_issue: Create a new issue in a GitHub repository
- create_pull_request: Create a new pull request in a GitHub repository
- fork_repository: Fork a GitHub repository to your account or specified organization
- create_branch: Create a new branch in a GitHub repository
- list_commits: Get list of commits of a branch in a GitHub repository
- list_issues: List issues in a GitHub

### 10.2.3 MCP 传输方式详解

MCP 协议具有**传输层无关性**（Transport Agnostic），可以在不同通信通道上运行。HelloAgents 基于 FastMCP 2.0 支持五种传输方式：

<div align="center">
  <p>表 10.4 MCP 传输方式对比</p>
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-table-4.png" alt="" width="85%"/>
</div>

In [7]:
import asyncio
from hello_agents.tools import MCPTool
from hello_agents.protocols import MCPClient

# ── 1. Memory Transport（内存传输，适合测试）──
mcp_tool = MCPTool()  # 不指定任何参数，使用内置演示服务器
result = mcp_tool.run({"action": "list_tools"})
print("Memory Transport 可用工具:", result)

result = mcp_tool.run({"action": "call_tool", "tool_name": "add", "arguments": {"a": 10, "b": 20}})
print(f"计算结果: {result}")

# ── 2. Stdio Transport（标准输入输出，适合本地开发）──
# mcp_tool = MCPTool(server_command=["python", "my_mcp_server.py"])
# mcp_tool = MCPTool(server_command=["npx", "-y", "@modelcontextprotocol/server-filesystem", "."])

# ── 3. HTTP Transport（远程服务，建议直接用MCPClient）──
async def test_http_transport():
    client = MCPClient("http://api.example.com/mcp")
    async with client:
        tools = await client.list_tools()
        print(f"远程服务器工具: {len(tools)} 个")
# asyncio.run(test_http_transport())

# ── 4. SSE Transport（实时通信）──
async def test_sse_transport():
    client = MCPClient("http://localhost:8080/sse", transport_type="sse")
    async with client:
        result = await client.call_tool("stream_process", {"input": "大量数据处理请求", "stream": True})
        print(f"流式处理结果: {result}")
# asyncio.run(test_sse_transport())

# ── 5. StreamableHTTP Transport（双向流式HTTP）──
async def test_streamable_http_transport():
    client = MCPClient("http://localhost:8080/mcp", transport_type="streamable_http")
    async with client:
        tools = await client.list_tools()
        print(f"StreamableHTTP 服务器工具: {len(tools)} 个")
# asyncio.run(test_streamable_http_transport())

INFO:mcp.server.lowlevel.server:Processing request of type ListToolsRequest
INFO:mcp.server.lowlevel.server:Processing request of type ListToolsRequest
INFO:mcp.server.lowlevel.server:Processing request of type CallToolRequest
INFO:mcp.server.lowlevel.server:Processing request of type ListToolsRequest


🧠 使用内存传输: HelloAgents-BuiltinServer
🔗 连接到 MCP 服务器...
✅ 连接成功！
🔌 连接已断开
🧠 使用内存传输: HelloAgents-BuiltinServer
🔗 连接到 MCP 服务器...
✅ 连接成功！
🔌 连接已断开
Memory Transport 可用工具: 找到 6 个工具:
- add: 加法计算器
- subtract: 减法计算器
- multiply: 乘法计算器
- divide: 除法计算器
- greet: 友好问候
- get_system_info: 获取系统信息

🧠 使用内存传输: HelloAgents-BuiltinServer
🔗 连接到 MCP 服务器...
✅ 连接成功！
🔌 连接已断开
计算结果: 工具 'add' 执行结果:
30.0


### 10.2.4 在智能体中使用 MCP 工具

`MCPTool` 有一个**自动展开**特性：当你添加一个 MCP 工具到 Agent 时，它会自动将 MCP 服务器提供的所有工具展开为独立工具。

In [8]:
from dotenv import load_dotenv
load_dotenv()

from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import MCPTool

# ── 方式1：使用内置演示服务器（最简单）──
agent = SimpleAgent(name="助手", llm=HelloAgentsLLM())
mcp_tool = MCPTool(name="calculator")  # 自动连接内置演示服务器
agent.add_tool(mcp_tool)
# ✅ MCP工具 'calculator' 已展开为 6 个独立工具：
#   calculator_add / calculator_subtract / calculator_multiply /
#   calculator_divide / calculator_greet / calculator_get_system_info

response = agent.run("计算 25 乘以 16")
print(response)

# ── 方式2：连接外部 MCP 服务器 ──
agent2 = SimpleAgent(name="文件助手", llm=HelloAgentsLLM())

fs_tool = MCPTool(
    name="filesystem",               # 指定唯一名称（用作工具前缀）
    description="访问本地文件系统",
    server_command=["npx", "-y", "@modelcontextprotocol/server-filesystem", "."]
)
agent2.add_tool(fs_tool)
# name="fs" → 展开为 fs_read_file、fs_write_file 等，避免命名冲突

# response = agent2.run("请读取README.md文件，并总结其中的主要内容")
# print(response)

INFO:mcp.server.lowlevel.server:Processing request of type ListToolsRequest


🧠 使用内存传输: HelloAgents-BuiltinServer
🔗 连接到 MCP 服务器...
✅ 连接成功！
🔌 连接已断开
✅ 工具 'calculator_add' 已注册。
✅ 工具 'calculator_subtract' 已注册。
✅ 工具 'calculator_multiply' 已注册。
✅ 工具 'calculator_divide' 已注册。
✅ 工具 'calculator_greet' 已注册。
✅ 工具 'calculator_get_system_info' 已注册。
✅ MCP工具 'calculator' 已展开为 6 个独立工具


INFO:mcp.server.lowlevel.server:Processing request of type CallToolRequest
INFO:mcp.server.lowlevel.server:Processing request of type ListToolsRequest


🧠 使用内存传输: HelloAgents-BuiltinServer
🔗 连接到 MCP 服务器...
✅ 连接成功！
🔌 连接已断开
25 乘以 16 的结果是 400。
📝 使用 Stdio 传输 (命令): npx -y @modelcontextprotocol/server-filesystem .
🔗 连接到 MCP 服务器...
✅ 连接成功！
🔌 连接已断开
✅ 工具 'filesystem_read_file' 已注册。
✅ 工具 'filesystem_read_text_file' 已注册。
✅ 工具 'filesystem_read_media_file' 已注册。
✅ 工具 'filesystem_read_multiple_files' 已注册。
✅ 工具 'filesystem_write_file' 已注册。
✅ 工具 'filesystem_edit_file' 已注册。
✅ 工具 'filesystem_create_directory' 已注册。
✅ 工具 'filesystem_list_directory' 已注册。
✅ 工具 'filesystem_list_directory_with_sizes' 已注册。
✅ 工具 'filesystem_directory_tree' 已注册。
✅ 工具 'filesystem_move_file' 已注册。
✅ 工具 'filesystem_search_files' 已注册。
✅ 工具 'filesystem_get_file_info' 已注册。
✅ 工具 'filesystem_list_allowed_directories' 已注册。
✅ MCP工具 'filesystem' 已展开为 14 个独立工具


In [9]:
"""
实战案例：多 Agent 协作的智能文档助手

- Agent1：GitHub搜索专家（使用 GitHub MCP 服务器）
- Agent2：文档生成专家（使用 文件系统 MCP 服务器）
"""
from dotenv import load_dotenv
load_dotenv()
import os
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import MCPTool

# Agent 1：GitHub搜索专家
github_searcher = SimpleAgent(
    name="GitHub搜索专家",
    llm=HelloAgentsLLM(),
    system_prompt="你是一个GitHub搜索专家。搜索GitHub仓库并返回清晰、结构化的结果（仓库名称、简短描述）。"
)
github_tool = MCPTool(name="gh", server_command=["npx", "-y", "@modelcontextprotocol/server-github"])
github_searcher.add_tool(github_tool)

# Agent 2：文档生成专家
document_writer = SimpleAgent(
    name="文档生成专家",
    llm=HelloAgentsLLM(),
    system_prompt="你是一个文档生成专家，根据提供信息生成结构化的Markdown报告（标题、简介、主要内容、总结）。"
)
fs_tool = MCPTool(name="fs", server_command=["npx", "-y", "@modelcontextprotocol/server-filesystem", "."])
document_writer.add_tool(fs_tool)

try:
    # 步骤1：搜索 GitHub
    search_results = github_searcher.run("搜索关于'AI agent'的GitHub仓库，返回前5个最相关的结果")
    print("搜索结果:", search_results[:300], "...")

    # 步骤2：生成报告
    report_content = document_writer.run(f"根据以下GitHub搜索结果生成Markdown研究报告：\n\n{search_results}")

    # 步骤3：保存报告
    with open("report.md", "w", encoding="utf-8") as f:
        f.write(report_content)
    print(f"✅ 报告已保存到 report.md（{os.path.getsize('report.md')} 字节）")
except Exception as e:
    print(f"❌ 错误: {e}")

🔑 自动加载环境变量: GITHUB_PERSONAL_ACCESS_TOKEN
📝 使用 Stdio 传输 (命令): npx -y @modelcontextprotocol/server-github
🔗 连接到 MCP 服务器...
✅ 连接成功！
🔌 连接已断开
✅ 工具 'gh_create_or_update_file' 已注册。
✅ 工具 'gh_search_repositories' 已注册。
✅ 工具 'gh_create_repository' 已注册。
✅ 工具 'gh_get_file_contents' 已注册。
✅ 工具 'gh_push_files' 已注册。
✅ 工具 'gh_create_issue' 已注册。
✅ 工具 'gh_create_pull_request' 已注册。
✅ 工具 'gh_fork_repository' 已注册。
✅ 工具 'gh_create_branch' 已注册。
✅ 工具 'gh_list_commits' 已注册。
✅ 工具 'gh_list_issues' 已注册。
✅ 工具 'gh_update_issue' 已注册。
✅ 工具 'gh_add_issue_comment' 已注册。
✅ 工具 'gh_search_code' 已注册。
✅ 工具 'gh_search_issues' 已注册。
✅ 工具 'gh_search_users' 已注册。
✅ 工具 'gh_get_issue' 已注册。
✅ 工具 'gh_get_pull_request' 已注册。
✅ 工具 'gh_list_pull_requests' 已注册。
✅ 工具 'gh_create_pull_request_review' 已注册。
✅ 工具 'gh_merge_pull_request' 已注册。
✅ 工具 'gh_get_pull_request_files' 已注册。
✅ 工具 'gh_get_pull_request_status' 已注册。
✅ 工具 'gh_update_pull_request_branch' 已注册。
✅ 工具 'gh_get_pull_request_comments' 已注册。
✅ 工具 'gh_get_pull_request_reviews' 已注册。
✅ MCP工具 '

KeyboardInterrupt: 

### 10.2.5 MCP 社区生态

MCP 协议的一个巨大优势是**丰富的社区生态**。三个主要资源库：

1. **Awesome MCP Servers**（https://github.com/punkpeye/awesome-mcp-servers）- 社区维护的精选列表
2. **MCP Servers Website**（https://mcpservers.org/）- 官方目录网站，带搜索功能  
3. **Official MCP Servers**（https://github.com/modelcontextprotocol/servers）- Anthropic 官方维护

<div align="center">
  <p>表 10.5 常用官方 MCP 服务器</p>
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-table-5.png" alt="" width="85%"/>
</div>

<div align="center">
  <p>表 10.6 社区热门 MCP 服务器</p>
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-table-6.png" alt="" width="85%"/>
</div>

In [10]:
from hello_agents.tools import MCPTool

# 1. 自动化网页测试（Playwright）
playwright_tool = MCPTool(
    name="playwright",
    server_command=["npx", "-y", "@playwright/mcp"]
)
# Agent可以自动：打开浏览器、填写表单、截图验证、生成测试报告

# 2. 智能笔记助手：搜索资讯（Perplexity）→ 整理笔记 → 保存到Obsidian
# 3. 项目管理自动化：GitHub Issue → Jira任务 → Sprint进度同步
# 4. 内容创作工作流：YouTube字幕 → 摘要 → 保存Notion → 播放Spotify
print("社区MCP服务器示例代码已展示，可根据需要取消注释运行")

📝 使用 Stdio 传输 (命令): npx -y @playwright/mcp
🔗 连接到 MCP 服务器...
✅ 连接成功！
🔌 连接已断开
社区MCP服务器示例代码已展示，可根据需要取消注释运行


## 10.3 A2A 协议实战

### 10.3.1 协议设计动机

MCP 解决了智能体与工具的交互，而 **A2A 解决智能体之间的协作问题**。

传统中央协调器（星型拓扑）的三大问题：
- **单点故障**：协调器失效导致系统瘫痪
- **性能瓶颈**：所有通信都经过中心节点
- **扩展困难**：增加智能体需要改动中心逻辑

A2A 采用**点对点（P2P）架构**，核心抽象是**任务（Task）**和**工件（Artifact）**：

<div align="center">
  <p>表 10.7 A2A 核心概念</p>
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-table-7.png" alt="" width="85%"/>
</div>

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-7.png" alt="" width="85%"/>
  <p>图 10.7 A2A 任务生命周期</p>
</div>

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-8.png" alt="" width="85%"/>
  <p>图 10.8 A2A 请求生命周期（代理发现→身份验证→消息API）</p>
</div>

### 10.3.2 使用 A2A 协议实战

In [1]:
from hello_agents.protocols.a2a.implementation import A2AServer, A2A_AVAILABLE

def create_calculator_agent():
    """创建一个计算器智能体"""
    if not A2A_AVAILABLE:
        print("❌ A2A SDK 未安装，请运行: pip install a2a-sdk")
        return None

    print("🧮 创建计算器智能体")

    # 创建 A2A 服务器
    calculator = A2AServer(
        name="calculator-agent",
        description="专业的数学计算智能体",
        version="1.0.0",
        capabilities={
            "math": ["addition", "subtraction", "multiplication", "division"],
            "advanced": ["power", "sqrt", "factorial"]
        }
    )

    # 添加基础计算技能，装饰器
    @calculator.skill("add")
    def add_numbers(query: str) -> str:
        """加法计算"""
        try:
            parts = query.replace("计算", "").replace("加", "+").replace("加上", "+")
            if "+" in parts:
                numbers = [float(x.strip()) for x in parts.split("+")]
                result = sum(numbers)
                return f"计算结果: {' + '.join(map(str, numbers))} = {result}"
            else:
                return "请使用格式: 计算 5 + 3"
        except Exception as e:
            return f"计算错误: {e}"

    @calculator.skill("multiply")
    def multiply_numbers(query: str) -> str:
        """乘法计算"""
        try:
            parts = query.replace("计算", "").replace("乘以", "*").replace("×", "*")
            if "*" in parts:
                numbers = [float(x.strip()) for x in parts.split("*")]
                result = 1
                for num in numbers:
                    result *= num
                return f"计算结果: {' × '.join(map(str, numbers))} = {result}"
            else:
                return "请使用格式: 计算 5 * 3"
        except Exception as e:
            return f"计算错误: {e}"

    @calculator.skill("info")
    def get_info(query: str) -> str:
        """获取智能体信息"""
        return f"我是 {calculator.name}，可以进行基础数学计算。支持的技能: {list(calculator.skills.keys())}"

    print(f"✅ 计算器智能体创建成功，支持技能: {list(calculator.skills.keys())}")
    return calculator

# 创建智能体
calc_agent = create_calculator_agent()
if calc_agent:
    # 测试技能
    print("\n🧪 测试智能体技能:")
    test_queries = [
        "获取信息",
        "计算 10 + 5",
        "计算 6 * 7"
    ]

    for query in test_queries:
        if "信息" in query:
            result = calc_agent.skills["info"](query)
        elif "+" in query:
            result = calc_agent.skills["add"](query)
        elif "*" in query or "×" in query:
            result = calc_agent.skills["multiply"](query)
        else:
            result = "未知查询类型"

        print(f"  📝 查询: {query}")
        print(f"  🤖 回复: {result}")
        print()


🧮 创建计算器智能体
✅ 计算器智能体创建成功，支持技能: ['add', 'multiply', 'info']

🧪 测试智能体技能:
  📝 查询: 获取信息
  🤖 回复: 我是 calculator-agent，可以进行基础数学计算。支持的技能: ['add', 'multiply', 'info']

  📝 查询: 计算 10 + 5
  🤖 回复: 计算结果: 10.0 + 5.0 = 15.0

  📝 查询: 计算 6 * 7
  🤖 回复: 计算结果: 6.0 × 7.0 = 42.0



In [2]:
from hello_agents.protocols.a2a.implementation import A2AServer, A2A_AVAILABLE

def create_custom_agent():
    """创建自定义智能体"""
    if not A2A_AVAILABLE:
        print("请先安装 A2A SDK: pip install a2a-sdk")
        return None

    # 创建智能体
    agent = A2AServer(
        name="my-custom-agent",
        description="我的自定义智能体",
        capabilities={"custom": ["skill1", "skill2"]}
    )

    # 添加技能
    @agent.skill("greet")
    def greet_user(name: str) -> str:
        """问候用户"""
        return f"你好，{name}！我是自定义智能体。"

    @agent.skill("calculate")
    def simple_calculate(expression: str) -> str:
        """简单计算"""
        try:
            # 安全的计算（仅支持基本运算）
            allowed_chars = set('0123456789+-*/(). ')
            if all(c in allowed_chars for c in expression):
                result = eval(expression)
                return f"计算结果: {expression} = {result}"
            else:
                return "错误: 只支持基本数学运算"
        except Exception as e:
            return f"计算错误: {e}"

    return agent

# 创建并测试自定义智能体
custom_agent = create_custom_agent()
if custom_agent:
    # 测试技能
    print("测试问候技能:")
    result1 = custom_agent.skills["greet"]("张三")
    print(result1)

    print("\n测试计算技能:")
    result2 = custom_agent.skills["calculate"]("10 + 5 * 2")
    print(result2)


测试问候技能:
你好，张三！我是自定义智能体。

测试计算技能:
计算结果: 10 + 5 * 2 = 20


### 10.3.3 使用 HelloAgents A2A 工具

HelloAgents 提供了统一的 A2A 工具接口，支持创建服务端、客户端以及多 Agent 网络。

In [3]:
from hello_agents.protocols import A2AServer
import threading
import time

# 创建研究员Agent服务
researcher = A2AServer(
    name="researcher",
    description="负责搜索和分析资料的Agent",
    version="1.0.0"
)

# 定义技能
@researcher.skill("research")
def handle_research(text: str) -> str:
    """处理研究请求"""
    import re
    match = re.search(r'research\s+(.+)', text, re.IGNORECASE)
    topic = match.group(1).strip() if match else text
    
    result = {
        "topic": topic,
        "findings": f"关于{topic}的研究结果...",
        "sources": ["来源1", "来源2", "来源3"]
    }
    return str(result)

# 在后台启动服务
def start_server():
    researcher.run(host="localhost", port=5000)

if __name__ == "__main__":
    server_thread = threading.Thread(target=start_server, daemon=True)
    server_thread.start()
    
    print("✅ 研究员Agent服务已启动在 http://localhost:5000")
    
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        print("\n服务已停止")


Exception in thread Thread-4 (start_server):
Traceback (most recent call last):
  File "/root/workspace/hello-agents/.venv/lib/python3.12/site-packages/hello_agents/protocols/a2a/implementation.py", line 62, in run
    from flask import Flask, request, jsonify
ModuleNotFoundError: No module named 'flask'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/tmp/ipykernel_325112/1072744102.py", line 29, in start_server
  File "/root/workspace/hello-agents/.venv/lib/python3.12/site-packages/hello_agents/protocols/a2a/implementation.py", line 64, in run
    raise ImportError(
ImportError: A2A server requires Flask. Install it with: pip install flask


✅ 研究员Agent服务已启动在 http://localhost:5000

服务已停止


In [4]:
from hello_agents.protocols import A2AClient

# 创建客户端连接到研究员Agent
client = A2AClient("http://localhost:5000")

# 发送研究请求
response = client.execute_skill("research", "research AI在医疗领域的应用")
print(f"收到响应：{response.get('result')}")


收到响应：None


In [ ]:
from hello_agents.protocols import A2AServer, A2AClient
import threading
import time

# 1. 创建多个Agent服务
researcher = A2AServer(name="researcher", description="研究员")

@researcher.skill("research")
def do_research(text: str) -> str:
    import re
    match = re.search(r'research\s+(.+)', text, re.IGNORECASE)
    topic = match.group(1).strip() if match else text
    return str({"topic": topic, "findings": f"{topic}的研究结果"})

writer = A2AServer(name="writer", description="撰写员")

@writer.skill("write")
def write_article(text: str) -> str:
    import re
    match = re.search(r'write\s+(.+)', text, re.IGNORECASE)
    content = match.group(1).strip() if match else text
    try:
        data = eval(content)
        topic = data.get("topic", "未知主题")
        findings = data.get("findings", "无研究结果")
    except:
        topic = "未知主题"
        findings = content
    return f"# {topic}\n\n基于研究：{findings}\n\n文章内容..."

editor = A2AServer(name="editor", description="编辑")

@editor.skill("edit")
def edit_article(text: str) -> str:
    import re
    match = re.search(r'edit\s+(.+)', text, re.IGNORECASE)
    article = match.group(1).strip() if match else text
    result = {"article": article + "\n\n[已编辑优化]", "feedback": "文章质量良好", "approved": True}
    return str(result)

# 2. 启动所有服务
threading.Thread(target=lambda: researcher.run(port=5000), daemon=True).start()
threading.Thread(target=lambda: writer.run(port=5001), daemon=True).start()
threading.Thread(target=lambda: editor.run(port=5002), daemon=True).start()
time.sleep(2)

# 3. 创建客户端连接到各个Agent
researcher_client = A2AClient("http://localhost:5000")
writer_client = A2AClient("http://localhost:5001")
editor_client = A2AClient("http://localhost:5002")

# 4. 协作流程
def create_content(topic):
    research = researcher_client.execute_skill("research", f"research {topic}")
    research_data = research.get('result', '')
    
    article = writer_client.execute_skill("write", f"write {research_data}")
    article_content = article.get('result', '')
    
    final = editor_client.execute_skill("edit", f"edit {article_content}")
    return final.get('result', '')

result = create_content("AI在医疗领域的应用")
print(f"\n最终结果：\n{result}")


### 10.3.4 在智能体中使用 A2A 工具

现在让我们看看如何将 A2A 集成到 HelloAgents 的智能体中。

In [ ]:
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import A2ATool
from dotenv import load_dotenv

load_dotenv()
llm = HelloAgentsLLM()

# 假设已经有一个研究员Agent服务运行在 http://localhost:5000

# 创建协调者Agent
coordinator = SimpleAgent(name="协调者", llm=llm)

# 添加A2A工具，连接到研究员Agent
researcher_tool = A2ATool(
    name="researcher",
    description="研究员Agent，可以搜索和分析资料",
    agent_url="http://localhost:5000"
)
coordinator.add_tool(researcher_tool)

# 协调者可以调用研究员Agent
response = coordinator.run("请让研究员帮我研究AI在教育领域的应用")
print(response)


In [ ]:
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import A2ATool
from hello_agents.protocols import A2AServer
import threading
import time
from dotenv import load_dotenv

load_dotenv()
llm = HelloAgentsLLM()

# 1. 创建技术专家Agent服务
tech_expert = A2AServer(name="tech_expert", description="技术专家，回答技术问题")

@tech_expert.skill("answer")
def answer_tech_question(text: str) -> str:
    import re
    match = re.search(r'answer\s+(.+)', text, re.IGNORECASE)
    question = match.group(1).strip() if match else text
    return f"技术回答：关于'{question}'，我建议您查看我们的技术文档..."

# 2. 创建销售顾问Agent服务
sales_advisor = A2AServer(name="sales_advisor", description="销售顾问，回答销售问题")

@sales_advisor.skill("answer")
def answer_sales_question(text: str) -> str:
    import re
    match = re.search(r'answer\s+(.+)', text, re.IGNORECASE)
    question = match.group(1).strip() if match else text
    return f"销售回答：关于'{question}'，我们有特别优惠..."

# 3. 启动服务
threading.Thread(target=lambda: tech_expert.run(port=6000), daemon=True).start()
threading.Thread(target=lambda: sales_advisor.run(port=6001), daemon=True).start()
time.sleep(2)

# 4. 创建接待员Agent
receptionist = SimpleAgent(
    name="接待员",
    llm=llm,
    system_prompt="""你是客服接待员，负责：
1. 分析客户问题类型（技术问题 or 销售问题）
2. 将问题转发给相应的专家
3. 整理专家的回答并返回给客户

请保持礼貌和专业。"""
)

# 添加工具
tech_tool = A2ATool(agent_url="http://localhost:6000", name="tech_expert", description="技术专家，回答技术相关问题")
sales_tool = A2ATool(agent_url="http://localhost:6001", name="sales_advisor", description="销售顾问，回答价格、购买相关问题")
receptionist.add_tool(tech_tool)
receptionist.add_tool(sales_tool)

# 5. 处理客户咨询
def handle_customer_query(query):
    print(f"\n客户咨询：{query}")
    print("=" * 50)
    response = receptionist.run(query)
    print(f"\n客服回复：{response}")
    print("=" * 50)

if __name__ == "__main__":
    handle_customer_query("你们的API如何调用？")
    handle_customer_query("企业版的价格是多少？")
    handle_customer_query("如何集成到我的Python项目中？")


In [ ]:
from hello_agents.protocols import A2AServer, A2AClient
import threading
import time

# 创建两个需要协商的Agent
agent1 = A2AServer(name="agent1", description="Agent 1")

@agent1.skill("propose")
def handle_proposal(text: str) -> str:
    """处理协商提案"""
    import re
    match = re.search(r'propose\s+(.+)', text, re.IGNORECASE)
    proposal_str = match.group(1).strip() if match else text
    try:
        proposal = eval(proposal_str)
        task = proposal.get("task")
        deadline = proposal.get("deadline")
        if deadline >= 7:
            result = {"accepted": True, "message": "接受提案"}
        else:
            result = {"accepted": False, "message": "时间太紧", "counter_proposal": {"deadline": 7}}
        return str(result)
    except:
        return str({"accepted": False, "message": "无效的提案格式"})

agent2 = A2AServer(name="agent2", description="Agent 2")

@agent2.skill("negotiate")
def negotiate_task(text: str) -> str:
    """发起协商"""
    import re
    match = re.search(r'negotiate\s+task:(.+?)\s+deadline:(\d+)', text, re.IGNORECASE)
    if match:
        task = match.group(1).strip()
        deadline = int(match.group(2))
        proposal = {"task": task, "deadline": deadline}
        return str({"status": "negotiating", "proposal": proposal})
    else:
        return str({"status": "error", "message": "无效的协商请求"})

# 启动服务
threading.Thread(target=lambda: agent1.run(port=7000), daemon=True).start()
threading.Thread(target=lambda: agent2.run(port=7001), daemon=True).start()
time.sleep(2)
print("✅ 协商Agent服务已启动")


## 10.4 ANP 协议实战

在 MCP 协议解决了工具调用、A2A 协议解决点对点智能体协作之后，ANP 协议则专注于解决大规模、开放网络环境下的智能体管理问题。

### 10.4.1 协议目标

当一个网络中存在大量功能各异的智能体时，系统会面临以下挑战：

- **服务发现**：如何快速找到能够处理该任务的智能体？
- **智能路由**：如何选择最合适的智能体（根据负载、成本等）？
- **动态扩展**：如何让新加入网络的智能体被其他成员发现和调用？

ANP 的设计目标就是提供一套标准化的机制，来解决服务发现、路由选择和网络扩展性问题。

<div align="center">
  <p>表 10.8 ANP 核心概念</p>
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-table-8.png" alt="" width="85%"/>
</div>

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-9.png" alt="" width="85%"/>
  <p>图 10.9 ANP 整体流程</p>
</div>

ANP 的主要流程包括：

**1. 服务的发现与匹配：** 智能体 A 通过公开的发现服务，基于语义或功能描述查询，定位到符合任务需求的智能体 B。发现服务通过预先爬取各智能体对外暴露的标准端点（`.well-known/agent-descriptions`）来建立索引。

**2. 基于 DID 的身份验证：** 智能体 A 使用其私钥对包含自身 DID 的请求进行签名，智能体 B 通过解析 DID 获取公钥验证签名真实性，从而建立可信通信。

**3. 标准化的服务执行：** 身份验证通过后，双方依据预定义的标准接口和数据格式进行数据交换或服务调用。标准化的交互流程是实现跨平台、跨系统互操作性的基础。

### 10.4.2 使用 ANP 服务发现

In [5]:
from hello_agents.protocols import ANPDiscovery, register_service

# 创建服务发现中心
discovery = ANPDiscovery()

# 注册Agent服务
register_service(
    discovery=discovery,
    service_id="nlp_agent_1",
    service_name="NLP处理专家A",
    service_type="nlp",
    capabilities=["text_analysis", "sentiment_analysis", "ner"],
    endpoint="http://localhost:8001",
    metadata={"load": 0.3, "price": 0.01, "version": "1.0.0"}
)

register_service(
    discovery=discovery,
    service_id="nlp_agent_2",
    service_name="NLP处理专家B",
    service_type="nlp",
    capabilities=["text_analysis", "translation"],
    endpoint="http://localhost:8002",
    metadata={"load": 0.7, "price": 0.02, "version": "1.1.0"}
)

print("✅ 服务注册完成")


✅ 服务注册完成


In [6]:
from hello_agents.protocols import discover_service

# 按类型查找
nlp_services = discover_service(discovery, service_type="nlp")
print(f"找到 {len(nlp_services)} 个NLP服务")

# 选择负载最低的服务
best_service = min(nlp_services, key=lambda s: s.metadata.get("load", 1.0))
print(f"最佳服务：{best_service.service_name} (负载: {best_service.metadata['load']})")


找到 2 个NLP服务
最佳服务：NLP处理专家A (负载: 0.3)


In [7]:
from hello_agents.protocols import ANPNetwork

# 创建网络
network = ANPNetwork(network_id="ai_cluster")

# 添加节点
for service in discovery.list_all_services():
    network.add_node(service.service_id, service.endpoint)

# 建立连接（根据能力匹配）
network.connect_nodes("nlp_agent_1", "nlp_agent_2")

stats = network.get_network_stats()
print(f"✅ 网络构建完成，共 {stats['total_nodes']} 个节点")


✅ 网络构建完成，共 2 个节点


### 10.4.3 实战案例

让我们构建一个完整的分布式任务调度系统：

In [8]:
from hello_agents.protocols import ANPDiscovery, register_service
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools.builtin import ANPTool
import random
from dotenv import load_dotenv

load_dotenv()
llm = HelloAgentsLLM()

# 1. 创建服务发现中心
discovery = ANPDiscovery()

# 2. 注册多个计算节点
for i in range(10):
    register_service(
        discovery=discovery,
        service_id=f"compute_node_{i}",
        service_name=f"计算节点{i}",
        service_type="compute",
        capabilities=["data_processing", "ml_training"],
        endpoint=f"http://node{i}:8000",
        metadata={
            "load": random.uniform(0.1, 0.9),
            "cpu_cores": random.choice([4, 8, 16]),
            "memory_gb": random.choice([16, 32, 64]),
            "gpu": random.choice([True, False])
        }
    )

print(f"✅ 注册了 {len(discovery.list_all_services())} 个计算节点")

# 3. 创建任务调度Agent
scheduler = SimpleAgent(
    name="任务调度器",
    llm=llm,
    system_prompt="""你是一个智能任务调度器，负责：
1. 分析任务需求
2. 选择最合适的计算节点
3. 分配任务

选择节点时考虑：负载、CPU核心数、内存、GPU等因素。"""
)

# 添加ANP工具
anp_tool = ANPTool(
    name="service_discovery",
    description="服务发现工具，可以查找和选择计算节点",
    discovery=discovery
)
scheduler.add_tool(anp_tool)

# 4. 智能任务分配
def assign_task(task_description):
    print(f"\n任务：{task_description}")
    print("=" * 50)
    response = scheduler.run(f"""
    请为以下任务选择最合适的计算节点：
    {task_description}

    要求：
    1. 列出所有可用节点
    2. 分析每个节点的特点
    3. 选择最合适的节点
    4. 说明选择理由
    """)
    print(response)
    print("=" * 50)

assign_task("训练一个大型深度学习模型，需要GPU支持")
assign_task("处理大量文本数据，需要高内存")
assign_task("运行轻量级数据分析任务")


✅ 注册了 10 个计算节点
✅ 工具 'service_discovery' 已注册。

任务：训练一个大型深度学习模型，需要GPU支持
好的，首先我将列出所有可用节点。接下来会分析每个节点的特点，包括负载、CPU核心数、内存、GPU等情况，然后给出最合适的节点和选择理由。

正在查询所有可用计算节点……

任务：处理大量文本数据，需要高内存
好的，首先我将列出所有可用计算节点，然后分析每个节点的内存、负载、CPU核心数等特点，最后选择最合适的节点并说明理由。

正在查询所有可用计算节点……

任务：运行轻量级数据分析任务
收到！由于服务发现工具执行时未指定 action 参数，导致无法获取可用节点信息。根据当前结果，无法完成以下操作：

1. 列出所有可用节点
2. 分析每个节点的特点
3. 选择最合适的节点
4. 说明选择理由

建议重新执行服务发现工具时，明确指定 action 参数（如 action=list 或 action=query），以便获取节点详细信息。获取节点数据后，才能进行合理分析和任务调度。

如果需要，我可以帮你构造正确的工具调用请求，请告知你需要查询的任务类型或节点资源，或者直接让我重新发起节点查询。


In [9]:
from hello_agents.protocols import ANPDiscovery, register_service
import random

# 创建服务发现中心
discovery = ANPDiscovery()

# 注册多个相同类型的服务
for i in range(5):
    register_service(
        discovery=discovery,
        service_id=f"api_server_{i}",
        service_name=f"API服务器{i}",
        service_type="api",
        capabilities=["rest_api"],
        endpoint=f"http://api{i}:8000",
        metadata={"load": random.uniform(0.1, 0.9)}
    )

# 负载均衡函数
def get_best_server():
    """选择负载最低的服务器"""
    servers = discovery.discover_services(service_type="api")
    if not servers:
        return None
    best = min(servers, key=lambda s: s.metadata.get("load", 1.0))
    return best

# 模拟请求分配
for i in range(10):
    server = get_best_server()
    print(f"请求 {i+1} -> {server.service_name} (负载: {server.metadata['load']:.2f})")
    # 更新负载（模拟）
    server.metadata["load"] += 0.1


请求 1 -> API服务器3 (负载: 0.31)
请求 2 -> API服务器0 (负载: 0.37)
请求 3 -> API服务器3 (负载: 0.41)
请求 4 -> API服务器0 (负载: 0.47)
请求 5 -> API服务器3 (负载: 0.51)
请求 6 -> API服务器0 (负载: 0.57)
请求 7 -> API服务器3 (负载: 0.61)
请求 8 -> API服务器1 (负载: 0.63)
请求 9 -> API服务器2 (负载: 0.66)
请求 10 -> API服务器0 (负载: 0.67)


## 10.5 构建自定义 MCP 服务器

在前面的章节中，我们学习了如何使用现有的 MCP 服务。现在，让我们学习如何构建自己的 MCP 服务器。

### 10.5.1 创建你的第一个 MCP 服务器

**为什么要构建自定义 MCP 服务器？**

虽然可以直接使用公开的 MCP 服务，但在许多实际应用场景中，需要构建自定义的 MCP 服务器以满足特定需求：

- **封装业务逻辑**：将企业内部特有的业务流程或复杂操作封装为标准化的 MCP 工具，供智能体统一调用。
- **访问私有数据**：创建一个安全可控的接口或代理，用于访问内部数据库、API 或其他无法对公网暴露的私有数据源。
- **性能专项优化**：针对高频调用或对响应延迟有严苛要求的应用场景，进行深度优化。
- **功能定制扩展**：实现标准 MCP 服务未提供的特定功能，例如集成专有算法模型或连接特定的硬件设备。

In [10]:
#!/usr/bin/env python3
"""天气查询 MCP 服务器"""

import json
import requests
import os
from datetime import datetime
from typing import Dict, Any
from hello_agents.protocols import MCPServer

# 创建 MCP 服务器
weather_server = MCPServer(name="weather-server", description="真实天气查询服务")

CITY_MAP = {
    "北京": "Beijing", "上海": "Shanghai", "广州": "Guangzhou",
    "深圳": "Shenzhen", "杭州": "Hangzhou", "成都": "Chengdu",
    "重庆": "Chongqing", "武汉": "Wuhan", "西安": "Xi'an",
    "南京": "Nanjing", "天津": "Tianjin", "苏州": "Suzhou"
}


def get_weather_data(city: str) -> Dict[str, Any]:
    """从 wttr.in 获取天气数据"""
    city_en = CITY_MAP.get(city, city)
    url = f"https://wttr.in/{city_en}?format=j1"
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    data = response.json() # 把返回的JSON字符串解析成Python字典
    current = data["current_condition"][0]

    return {
        "city": city,
        "temperature": float(current["temp_C"]),
        "feels_like": float(current["FeelsLikeC"]),
        "humidity": int(current["humidity"]),
        "condition": current["weatherDesc"][0]["value"],
        "wind_speed": round(float(current["windspeedKmph"]) / 3.6, 1),
        "visibility": float(current["visibility"]),
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }


def get_weather(city: str) -> str:
    """获取指定城市的当前天气"""
    try:
        weather_data = get_weather_data(city)
        return json.dumps(weather_data, ensure_ascii=False, indent=2)
        #将Python字典转换成JSON字符串，ensure_ascii=False允许中文正常显示，indent=2美化输出   
    except Exception as e:
        return json.dumps({"error": str(e), "city": city}, ensure_ascii=False)


def list_supported_cities() -> str:
    """列出所有支持的中文城市"""
    result = {"cities": list(CITY_MAP.keys()), "count": len(CITY_MAP)}
    return json.dumps(result, ensure_ascii=False, indent=2)


def get_server_info() -> str:
    """获取服务器信息"""
    info = {
        "name": "Weather MCP Server",
        "version": "1.0.0",
        "tools": ["get_weather", "list_supported_cities", "get_server_info"]
    }
    return json.dumps(info, ensure_ascii=False, indent=2)


# 注册工具到服务器
weather_server.add_tool(get_weather)
weather_server.add_tool(list_supported_cities)
weather_server.add_tool(get_server_info)


if __name__ == "__main__":
    weather_server.run()


RuntimeError: Already running asyncio in this thread

In [ ]:
#!/usr/bin/env python3
"""测试天气查询 MCP 服务器"""

import asyncio
import json
import sys
import os

sys.path.insert(0, os.path.join(os.path.dirname(__file__), '..', 'HelloAgents'))
from hello_agents.protocols.mcp.client import MCPClient


async def test_weather_server():
    server_script = os.path.join(os.path.dirname(__file__), "14_weather_mcp_server.py")
    client = MCPClient(["python", server_script])

    try:
        async with client:
            # 测试1: 获取服务器信息
            info = json.loads(await client.call_tool("get_server_info", {}))
            print(f"服务器: {info['name']} v{info['version']}")

            # 测试2: 列出支持的城市
            cities = json.loads(await client.call_tool("list_supported_cities", {}))
            print(f"支持城市: {cities['count']} 个")

            # 测试3: 查询北京天气
            weather = json.loads(await client.call_tool("get_weather", {"city": "北京"}))
            if "error" not in weather:
                print(f"\n北京天气: {weather['temperature']}°C, {weather['condition']}")

            # 测试4: 查询深圳天气
            weather = json.loads(await client.call_tool("get_weather", {"city": "深圳"}))
            if "error" not in weather:
                print(f"深圳天气: {weather['temperature']}°C, {weather['condition']}")

            print("\n✅ 所有测试完成！")

    except Exception as e:
        print(f"❌ 测试失败: {e}")


if __name__ == "__main__":
    asyncio.run(test_weather_server())


NameError: name '__file__' is not defined

In [12]:
"""在 Agent 中使用天气 MCP 服务器"""

import os
from dotenv import load_dotenv
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools import MCPTool

load_dotenv()


def create_weather_assistant():
    """创建天气助手"""
    llm = HelloAgentsLLM()

    assistant = SimpleAgent(
        name="天气助手",
        llm=llm,
        system_prompt="""你是天气助手，可以查询城市天气。
使用 get_weather 工具查询天气，支持中文城市名。
"""
    )

    # 添加天气 MCP 工具
    server_script = os.path.join(os.path.dirname(__file__), "14_weather_mcp_server.py")
    weather_tool = MCPTool(server_command=["python", server_script])
    assistant.add_tool(weather_tool)

    return assistant


def demo():
    """演示"""
    assistant = create_weather_assistant()

    print("\n查询北京天气：")
    response = assistant.run("北京今天天气怎么样？")
    print(f"回答: {response}\n")


def interactive():
    """交互模式"""
    assistant = create_weather_assistant()

    while True:
        user_input = input("\n你: ").strip()
        if user_input.lower() in ['quit', 'exit']:
            break
        response = assistant.run(user_input)
        print(f"助手: {response}")


if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1 and sys.argv[1] == "demo":
        demo()
    else:
        interactive()


NameError: name '__file__' is not defined

### 10.5.2 上传 MCP 服务器

我们创建了一个真实的天气查询 MCP 服务器。现在，让我们将它发布到 Smithery 平台，让全世界的开发者都能使用我们的服务。

**什么是 Smithery？**

[Smithery](https://smithery.ai/) 是 MCP 服务器的官方发布平台，类似于 Python 的 PyPI 或 Node.js 的 npm。通过 Smithery，用户可以：

- 🔍 发现和搜索 MCP 服务器
- 📦 一键安装 MCP 服务器
- 📊 查看服务器的使用统计和评价
- 🔄 自动获取服务器更新

**准备发布**

首先，我们需要将项目整理成标准的发布格式：

```
weather-mcp-server/
├── README.md           # 项目说明文档
├── LICENSE             # 开源许可证
├── Dockerfile          # Docker 构建配置（推荐）
├── pyproject.toml      # Python 项目配置（必需）
├── requirements.txt    # Python 依赖
├── smithery.yaml       # Smithery 配置文件（必需）
└── server.py           # MCP 服务器主文件
```

`smithery.yaml` 是 Smithery 平台的配置文件：

```yaml
name: weather-mcp-server
displayName: Weather MCP Server
description: Real-time weather query MCP server based on HelloAgents framework
version: 1.0.0
author: HelloAgents Team
homepage: https://github.com/yourusername/weather-mcp-server
license: MIT
categories:
  - weather
  - data
tags:
  - weather
  - real-time
  - helloagents
  - wttr
runtime: container
build:
  dockerfile: Dockerfile
  dockerBuildPath: .
startCommand:
  type: http
tools:
  - name: get_weather
    description: Get current weather for a city
  - name: list_supported_cities
    description: List all supported cities
  - name: get_server_info
    description: Get server information
```

`pyproject.toml` 是 Python 项目的标准配置文件，Smithery 要求必须包含此文件：

```toml
[build-system]
requires = ["setuptools>=61.0", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "weather-mcp-server"
version = "1.0.0"
description = "Real-time weather query MCP server based on HelloAgents framework"
readme = "README.md"
license = {text = "MIT"}
authors = [
    {name = "HelloAgents Team", email = "xxx"}
]
requires-python = ">=3.10"
dependencies = [
    "hello-agents>=0.2.1",
    "requests>=2.31.0",
]

[project.urls]
Homepage = "https://github.com/yourusername/weather-mcp-server"
Repository = "https://github.com/yourusername/weather-mcp-server"

[tool.setuptools]
py-modules = ["server"]
```

`Dockerfile` 确保部署成功：

```dockerfile
FROM python:3.12-slim-bookworm as base

WORKDIR /app

RUN apt-get update && apt-get install -y \
    --no-install-recommends \
    && rm -rf /var/lib/apt/lists/*

COPY pyproject.toml requirements.txt ./
COPY server.py ./

RUN pip install --no-cache-dir --upgrade pip && \
    pip install --no-cache-dir -r requirements.txt

ENV PYTHONUNBUFFERED=1
ENV PORT=8081

EXPOSE 8081

HEALTHCHECK --interval=30s --timeout=3s --start-period=5s --retries=3 \
    CMD python -c "import sys; sys.exit(0)"

CMD ["python", "server.py"]
```

**提交到 Smithery**

打开浏览器访问 [https://smithery.ai/](https://smithery.ai/)，使用 GitHub 账号登录，点击 "Publish Server" 按钮，输入你的 GitHub 仓库 URL，等待发布。

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-10.png" alt="" width="85%"/>
  <p>图 10.10 Smithery 发布成功页面</p>
</div>

发布成功后，用户可以通过以下方式使用：

```python
from hello_agents import SimpleAgent, HelloAgentsLLM
from hello_agents.tools.builtin.protocol_tools import MCPTool

agent = SimpleAgent(name="天气助手", llm=HelloAgentsLLM())

# 使用 Smithery 安装的服务器
weather_tool = MCPTool(
    server_command=["smithery", "run", "weather-mcp-server"]
)
agent.add_tool(weather_tool)

response = agent.run("北京今天天气怎么样？")
```

<div align="center">
  <img src="https://raw.githubusercontent.com/datawhalechina/Hello-Agents/main/docs/images/10-figures/10-11.png" alt="" width="85%"/>
  <p>图 10.11 Smithery 发布成功的 MCP 工具</p>
</div>

如果想要更加深入了解，可以点击这个[链接](https://smithery.ai/server/@jjyaoao/weather-mcp-server)。

## 10.6 本章总结

本章系统性地介绍了智能体通信的三种核心协议：MCP、A2A 与 ANP，并探讨了它们的设计理念、应用场景与实践方法。

**协议定位：**

- **MCP (Model Context Protocol)**: 作为智能体与工具之间的桥梁，提供统一的工具访问接口，适用于增强单个智能体的能力。
- **A2A (Agent-to-Agent Protocol)**: 作为智能体之间的对话系统，支持直接通信与任务协商，适用于小规模团队的紧密协作。
- **ANP (Agent Network Protocol)**: 作为智能体的"互联网"，提供服务发现、路由与负载均衡机制，适用于构建大规模、开放的智能体网络。

**HelloAgents 的集成方案**

在 `HelloAgents` 框架中，这三种协议被统一抽象为工具（Tool），实现了无缝集成：

```python
# 统一的Tool接口
from hello_agents.tools import MCPTool, A2ATool, ANPTool

# 所有协议都可以作为Tool添加到Agent
agent.add_tool(MCPTool(...))
agent.add_tool(A2ATool(...))
agent.add_tool(ANPTool(...))
```

**实战经验总结**

- 优先利用成熟的社区 MCP 服务，以减少不必要的重复开发。
- 根据系统规模选择合适的协议：小规模协作场景推荐使用 A2A，大规模网络场景则应采用 ANP。

完成本章后，建议你：

1. **动手实践**：
   - 构建自己的 MCP 服务器
   - 利用协议创建多 Agent 协作系统
   - MCP、A2A 与 ANP 的组合应用策略

2. **深入学习**：
   - 阅读 MCP 官方文档：https://modelcontextprotocol.io
   - 阅读 A2A 官方文档：https://a2a-protocol.org/latest/
   - 阅读 ANP 官方文档：https://agent-network-protocol.com/guide/

3. **参与社区**：
   - 向社区贡献新的 MCP 服务
   - 分享个人开发的智能体实现案例
   - 参与相关协议的技术标准讨论，也可以在 Issue 提问或是直接帮助 HelloAgents 支持新的 example 案例

**恭喜你完成第十章的学习！**

你现在已经掌握了智能体通信协议的核心知识。继续加油！🚀

---

## 习题

> **提示**：部分习题没有标准答案，重点在于培养学习者对智能体通信协议的综合理解和实践能力。

**1. 协议对比与选型**

本章介绍了三种智能体通信协议：MCP、A2A 和 ANP。请分析：

- 为什么 MCP 强调"上下文共享"，A2A 强调"对话式协作"，而 ANP 强调"网络拓扑"？这些设计理念分别解决了什么核心问题？
- 假设你要构建一个"智能客服系统"，需要以下功能：（1）访问客户数据库和订单系统；（2）多个专业客服智能体协作处理复杂问题；（3）支持大规模并发用户请求。请为每个功能选择最合适的协议，并说明理由。
- 三种协议是否可以组合使用？请设计一个实际应用场景，展示如何同时使用 MCP、A2A 和 ANP 来构建一个完整的智能体系统。

用户提交创作需求
        │
        ▼
  ┌─────────────────────────────────────────┐
  │           ANP 服务发现层                 │
  │  发现当前负载最低的"内容协调Agent"节点    │
  └─────────────┬───────────────────────────┘
                │
                ▼
  ┌─────────────────────────────────────────┐
  │        内容协调Agent（接待员）            │
  │  通过 A2A 与各专业Agent点对点通信         │
  └──────┬──────────┬──────────┬────────────┘
         │          │          │
    A2A  ▼     A2A  ▼     A2A  ▼
  研究员Agent  写作Agent  审稿Agent
      │            │          │
  MCP ▼        MCP ▼      MCP ▼
 搜索引擎MCP  文件系统MCP  语法检查MCP
 (外部数据)   (保存文件)   (外部工具)

**2. MCP 深入实践（动手题）**

基于 10.2 节的内容，请深入思考：

- 在 10.2.3 节的 MCP 服务器实现中，请扩展这个实现，添加一个新的 MCP 服务器，提供以下工具：（1）数据库查询工具；（2）数据可视化工具；（3）报表生成工具。要求工具之间能够协作完成复杂的数据分析任务。
- MCP 协议支持"资源"（Resources）和"提示"（Prompts）两个重要概念，请查阅 MCP 官方文档，了解 Resources 和 Prompts 的设计目的，并设计一个应用场景展示如何利用这三个核心概念构建更强大的智能体系统。
- MCP 使用 JSON-RPC 2.0 作为底层通信协议，通过 stdio 进行进程间通信。请分析：这种设计有什么优势和局限性？如果需要支持远程 MCP 服务器（通过 HTTP/WebSocket 访问），应该如何扩展当前的实现？

                    ┌──────────────────┐
                    │   代码审查 MCP    │
                    └────────┬─────────┘
                             │
        ┌────────────────────┼────────────────────┐
        ▼                    ▼                    ▼
   📦 Resources          🔧 Tools            💬 Prompts
   (提供只读上下文)      (主动执行操作)      (标准化任务模板)
        │                    │                    │
  代码风格规范文档      run_linter(file)     code_review
  历史Bug数据库        run_tests(file)      (language, style)
  项目依赖树           add_comment(...)     security_audit
                       create_pr(...)       (severity_level)

任务类型
    │
    ├─ 本地开发 / 工具在同一台机器 ──→  stdio（零配置，推荐默认）
    │
    ├─ 跨机器调用 / 微服务部署      ──→  HTTP（无状态，易扩展）
    │
    └─ 实时流式 / 长对话 / 推送通知 ──→  WebSocket / SSE（双向通信）



**3. A2A 协作扩展（动手题）**

基于 10.3 节的内容，请完成以下扩展实践：

- 在 10.3.4 节的"研究团队"案例中，请扩展这个案例，添加第三个智能体"审稿人"（Reviewer），它能够评审论文质量并提出修改建议。设计三个智能体之间的协作流程，并实现完整的代码。
- A2A 协议定义了 `task`、`task_result` 等消息类型。请分析：如果协作过程中出现冲突，应该如何设计冲突解决机制？请扩展 A2A 协议，添加"协商"（negotiation）和"投票"（voting）等消息类型。
- 对比 A2A 协议与第六章介绍的 AutoGen、CAMEL 等多智能体框架：A2A 作为标准协议，与这些框架的关系是什么？它们能否互相替代？

**4. ANP 网络设计**

基于 10.4 节的内容，请深入分析：

- 在什么场景下应该选择不同的网络拓扑结构（星型、网状、分层）？如果网络规模从 10 个智能体扩展到 1000 个智能体，拓扑结构应该如何演进？

节点数量        推荐拓扑        原因
─────────────────────────────────────────────────
≤ 20 个        星型            简单易维护，协调者管理全局
20 ~ 50 个     网状            去中心化，容错性强
50 ~ 200 个    分层（2层）     分组管理，平衡性能与复杂度
200 ~ 1000 个  分层（3层）     必须分层，否则路由表过大
> 1000 个      分层 + 动态发现  类似互联网BGP路由，各区域自治

- 请设计一个"智能路由算法"：根据任务类型、智能体能力、网络负载等因素，自动选择最优的消息路由路径。


- 如果某个关键智能体出现故障，整个系统应该如何应对？请设计一个"容错机制"，包括故障检测、备份切换、状态恢复等功能。

正常运行
  │
  ├─ 定期心跳检测 ──→ 超时/异常 ──→ 标记为"疑似故障"
  │                                        │
  │                                   重试N次确认
  │                                        │
  │                              ┌─────────┴──────────┐
  │                           恢复正常              确认故障
  │                           取消标记           ↓
  │                                      切换备份节点
  │                                      恢复中断任务
  │                                      通知相关节点
  │                                      异步尝试修复原节点
  └─────────────────────────────────────────────────────

**5. 安全与隐私**

智能体通信协议的安全性和隐私保护是实际应用中的关键问题。请思考：

- 在 MCP 客户端实现中，如果 MCP 服务器提供了危险操作（如删除文件、执行系统命令），应该如何设计权限控制机制？

"我会设计三道防线：
第一层是工具白名单，客户端只暴露预审批的工具列表给 Agent，危险工具（比如 exec_shell、delete_file）默认不在列表里，Agent 根本看不到它们；
第二层是参数过滤，即使工具被允许调用，也要校验参数，比如文件路径不能包含 ../，命令不能有管道符；
第三层是危险操作二次确认，对于删除这类不可逆操作，必须弹出人工确认，不能让 Agent 单独决策。"

- A2A 和 ANP 协议涉及多个智能体之间的通信，可能包含敏感信息。请设计一个"端到端加密"方案，确保消息在传输过程中不被窃听或篡改。


- 在大规模智能体网络中，请设计一个"信任评估系统"：根据智能体的历史行为、协作质量、社区评价等因素，动态评估每个智能体的可信度。

---

## 参考文献

[1] Anthropic. (2024). *Model Context Protocol*. Retrieved October 7, 2025, from https://modelcontextprotocol.io/

[2] The A2A Project. (2025). *A2A Protocol: An open protocol for agent-to-agent communication*. Retrieved October 7, 2025, from https://a2a-protocol.org/

[3] Chang, G., Lin, E., Yuan, C., Cai, R., Chen, B., Xie, X., & Zhang, Y. (2025). *Agent Network Protocol technical white paper*. arXiv. https://doi.org/10.48550/arXiv.2508.00007